# Example 9.4 / 1: Read One File per Period

Companion notebook for **9.4 Indexed Tables and Spreadsheet Sheets** in the AMPL lab Documentation, section "Read One File per Period". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read One File per Period

Each period's sales limits are in their own file, `market_1.csv` and `market_2.csv`, with the
same two columns `product` and `market`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
set PERIOD;                       # planning periods, one file each
param market {PROD, PERIOD} >= 0; # sales limit by product and period

table PeriodMarket {t in PERIOD} IN "amplcsv" ("market_" & t & ".csv"):
    [p ~ product], market[p, t] ~ market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("market_1.csv", "w") as f:
    f.write("product,market\nbands,6000\ncoils,4000\nplate,3500\n")
with open("market_2.csv", "w") as f:
    f.write("product,market\nbands,5000\ncoils,4500\nplate,3000\n")
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["PERIOD"] = [1, 2]

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
for t in ampl.set["PERIOD"].to_list():
    ampl.read_table(f"PeriodMarket[{t}]")
print(ampl.param["market"].to_pandas())

               market
index0 index1        
bands  1         6000
       2         5000
coils  1         4000
       2         4500
plate  1         3500
       2         3000


### Pure AMPL: data.dat and model.run

The parenthesized string expression is evaluated for each `t`, giving the file name. The key
column `product` is renamed to the dummy `p` with `p ~ product`, so the data column can say
which period it fills: `market[p, t]`. Every member of the family uses the same headings; only
the file changes.

Pure AMPL, with the same `model.mod` and files:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set PERIOD := 1 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
for {t in PERIOD} read table PeriodMarket[t];
display market;

market :=
bands 1   6000
bands 2   5000
coils 1   4000
coils 2   4500
plate 1   3500
plate 2   3000
;

